# 01 按厂商/产品系列的嵌套分组验证

本 Notebook 默认从仓库的 `Data/Actuator_Product_Dataset.xlsx` 读取公开数据，并把新结果写入 `Reproduced_Outputs/`。路径覆盖方式见 `Code/README.md`。

In [ ]:
from pathlib import Path
from datetime import datetime
import os, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, recall_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold, GroupKFold
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore")
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
# Optional overrides. Leave unset when Jupyter starts in the repository root or Code/.
REPO_ROOT_OVERRIDE = os.getenv("ROBOT_ACTUATOR_REPO_ROOT") or None
DATASET_PATH_OVERRIDE = os.getenv("ROBOT_ACTUATOR_DATASET_PATH") or None
OUTPUT_DIR_OVERRIDE = os.getenv("ROBOT_ACTUATOR_OUTPUT_DIR") or None

def resolve_repo_root(override=None):
    if override:
        root = Path(override).expanduser().resolve()
        if not (root / "Code").is_dir() or not (root / "Data").is_dir():
            raise FileNotFoundError(f"仓库根目录无效：{root}")
        return root
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "Code").is_dir() and (candidate / "Data").is_dir():
            return candidate
    raise FileNotFoundError(
        "无法自动定位仓库根目录。请从仓库根目录/Code 启动 Jupyter，"
        "或设置 ROBOT_ACTUATOR_REPO_ROOT。"
    )

REPO_ROOT = resolve_repo_root(REPO_ROOT_OVERRIDE)
DATASET_PATH = (
    Path(DATASET_PATH_OVERRIDE).expanduser().resolve()
    if DATASET_PATH_OVERRIDE
    else REPO_ROOT / "Data" / "Actuator_Product_Dataset.xlsx"
)
OUTPUT_DIR = (
    Path(OUTPUT_DIR_OVERRIDE).expanduser().resolve()
    if OUTPUT_DIR_OVERRIDE
    else REPO_ROOT / "Reproduced_Outputs" / "01_grouped_nested_validation"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 42
TARGET_ORDER = ["EMR", "QDD", "HCL", "HRA"]

def normal(s):
    return re.sub(r"[^a-z0-9\u4e00-\u9fff]", "", str(s).lower())

def find_col(cols, options, required=True):
    lookup = {normal(c): c for c in cols}
    for x in options:
        if normal(x) in lookup: return lookup[normal(x)]
    for c in cols:
        if any(normal(x) in normal(c) for x in options): return c
    if required: raise KeyError(f"未找到列 {options}；实际列：{list(cols)}")
    return None

if not DATASET_PATH.is_file():
    raise FileNotFoundError(
        f"未找到公开数据集：{DATASET_PATH}。请检查仓库是否完整，或设置 ROBOT_ACTUATOR_DATASET_PATH。"
    )
raw = pd.read_excel(
    DATASET_PATH,
    sheet_name="Actuator_Product_Dataset",
    header=3,
)
EXCEL_PATH = DATASET_PATH
TARGET_COL = find_col(raw.columns, ["驱动形式", "驱动类型", "drive_type", "label", "类别"])
FEATURE_COLS = [
    find_col(raw.columns, ["输出角度", "angle"]),
    find_col(raw.columns, ["额定转速", "rated speed"]),
    find_col(raw.columns, ["峰值转速", "peak speed"]),
    find_col(raw.columns, ["额定力矩", "rated torque"]),
    find_col(raw.columns, ["峰值力矩", "peak torque"]),
    find_col(raw.columns, ["额定功率", "rated power"]),
    find_col(raw.columns, ["峰值功率", "peak power"]),
]
MANUFACTURER_COL = find_col(raw.columns, ["厂商", "制造商", "manufacturer", "品牌"], False)
MODEL_COL = find_col(raw.columns, ["型号", "model"], False)
FAMILY_COL = find_col(raw.columns, ["产品系列", "系列", "family"], False)
df = raw.copy()
df[TARGET_COL] = df[TARGET_COL].astype(str).str.strip().str.upper()
df = df[df[TARGET_COL].isin(TARGET_ORDER)].reset_index(drop=True)
for col in FEATURE_COLS: df[col] = pd.to_numeric(df[col], errors="coerce")
X = df[FEATURE_COLS]
encoder = LabelEncoder().fit(TARGET_ORDER)
y = encoder.transform(df[TARGET_COL])
CLASS_NAMES = list(encoder.classes_)
print(f"数据：{EXCEL_PATH}; 样本数={len(df)}; 类别={df[TARGET_COL].value_counts().to_dict()}")

def make_model():
    try:
        from xgboost import XGBClassifier
        model = XGBClassifier(objective="multi:softprob", eval_metric="mlogloss", n_estimators=350,
            max_depth=3, learning_rate=0.05, subsample=.9, colsample_bytree=.9,
            random_state=RANDOM_STATE, n_jobs=-1)
        backend = "XGBoost"
    except ImportError:
        model = RandomForestClassifier(n_estimators=600, class_weight="balanced",
            random_state=RANDOM_STATE, n_jobs=-1)
        backend = "RandomForest fallback; 正式实验请安装 xgboost 后重跑"
    return Pipeline([("imputer", SimpleImputer(strategy="median")), ("model", model)]), backend

def full_proba(model, part):
    p = np.asarray(model.predict_proba(part))
    classes = np.asarray(
        getattr(
            model,
            "_ec_global_classes_",
            getattr(model.named_steps["model"], "_ec_global_classes_", model.named_steps["model"].classes_),
        ),
        dtype=int,
    )
    out = np.zeros((len(part), len(CLASS_NAMES)))
    cols = min(p.shape[1], len(classes))
    out[:, classes[:cols]] = p[:, :cols]
    return out / np.maximum(out.sum(axis=1, keepdims=True), 1e-12)

def topk(y_true, p, k):
    return np.mean([t in np.argsort(row)[-k:] for t, row in zip(y_true, p)])

def fit_model(model, Xtr, ytr):
    ytr = np.asarray(ytr, dtype=int)
    classes = np.unique(ytr)
    y_local = np.zeros(len(ytr), dtype=int) if len(classes) == 1 else np.searchsorted(classes, ytr)
    model.fit(Xtr, y_local)
    setattr(model, "_ec_global_classes_", classes)
    if hasattr(model, "named_steps") and "model" in model.named_steps:
        setattr(model.named_steps["model"], "_ec_global_classes_", classes)
    return model

def predict_global(model, part):
    pred = np.asarray(model.predict(part), dtype=int)
    classes = np.asarray(
        getattr(
            model,
            "_ec_global_classes_",
            getattr(model.named_steps["model"], "_ec_global_classes_", model.named_steps["model"].classes_),
        ),
        dtype=int,
    )
    pred = np.clip(pred, 0, len(classes) - 1)
    return classes[pred]

In [ ]:
def build_group_ids(frame):
    n = len(frame); parent = list(range(n))
    def root(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]; i = parent[i]
        return i
    def join(a, b):
        a, b = root(a), root(b)
        if a != b: parent[b] = a
    maker = frame[MANUFACTURER_COL].fillna("unknown").astype(str).str.strip() if MANUFACTURER_COL else pd.Series("unknown", index=frame.index)
    if FAMILY_COL:
        family = frame[FAMILY_COL].fillna("").astype(str).str.strip()
    elif MODEL_COL:
        family = frame[MODEL_COL].fillna("").astype(str).str.upper().str.replace(r"[0-9].*$", "", regex=True)
    else:
        family = pd.Series("", index=frame.index)
    # 同系列和七维性能完全相同的样本通过并查集合并，避免信息泄漏。
    family_key = (maker + "|" + family).where(family.ne(""), "")
    duplicate_key = frame[FEATURE_COLS].round(8).astype(str).agg("|".join, axis=1)
    for key_series in (family_key, duplicate_key):
        seen = {}
        for i, key in enumerate(key_series):
            if key in seen and key: join(i, seen[key])
            elif key: seen[key] = i
    roots = [root(i) for i in range(n)]
    remap = {r: f"G{i:03d}" for i, r in enumerate(sorted(set(roots)))}
    group = np.array([remap[r] for r in roots])
    audit = pd.DataFrame({"manufacturer_id": maker, "family_id": family,
        "duplicate_cluster_id": duplicate_key, "group_id": group, "drive_type": frame[TARGET_COL]})
    return group, audit

groups, group_audit = build_group_ids(df)
group_counts = pd.DataFrame({"y": y, "g": groups}).groupby("y").g.nunique()
N_SPLITS = min(5, int(group_counts.min()))
if N_SPLITS < 3: raise ValueError(f"最小类别仅 {N_SPLITS} 个独立组；请人工补充 family_id 后再运行。")
print("每类独立组数：", group_counts.to_dict(), "；外层分组折数：", N_SPLITS)

group_audit.to_csv(OUTPUT_DIR / "group_assignment_audit.csv", index=False, encoding="utf-8-sig")


## 内层调参与无泄漏评估

缺失值处理、调参均在训练折内完成。内层使用 3 折分组验证，以 Macro-F1 选择参数；若训练侧独立组不足则保守使用默认参数并记录。


In [ ]:
PARAM_GRID = [
 {"max_depth":d, "learning_rate":lr, "subsample":ss, "colsample_bytree":cs}
 for d in [2,3,4] for lr in [.03,.06] for ss in [.8,1.] for cs in [.8,1.]
][:12]

def model_with(params):
    try:
        from xgboost import XGBClassifier
        m = XGBClassifier(objective="multi:softprob", eval_metric="mlogloss", n_estimators=350,
            random_state=RANDOM_STATE, n_jobs=-1, **params)
        return Pipeline([("imputer", SimpleImputer(strategy="median")), ("model",m)])
    except ImportError:
        return make_model()[0]

def select_params(Xtr, ytr, gtr):
    unique = pd.DataFrame({"y":ytr, "g":gtr}).groupby("y").g.nunique()
    n_inner = min(3, int(unique.min()))
    if n_inner < 2: return {}, np.nan
    cv = StratifiedGroupKFold(n_splits=n_inner, shuffle=True, random_state=RANDOM_STATE)
    best, score = {}, -np.inf
    for params in PARAM_GRID:
        scores=[]
        for a,b in cv.split(Xtr, ytr, gtr):
            m=model_with(params); fit_model(m, Xtr.iloc[a], ytr[a])
            scores.append(f1_score(ytr[b],predict_global(m,Xtr.iloc[b]),average="macro",zero_division=0))
        if np.mean(scores) > score: best,score=params,float(np.mean(scores))
    return best,score

def evaluate(cv, grouped=False, nested=False, label=""):
    folds=[]; records=[]
    iterator = cv.split(X,y,groups) if grouped else cv.split(X,y)
    for fold,(tr,te) in enumerate(iterator,1):
        params, inner = select_params(X.iloc[tr],y[tr],groups[tr]) if nested else ({},np.nan)
        m = model_with(params) if nested else make_model()[0]
        fit_model(m, X.iloc[tr], y[tr]); p=full_proba(m,X.iloc[te]); pred=p.argmax(1)
        row={"protocol":label,"fold":fold,"inner_macro_f1":inner,
             "accuracy":accuracy_score(y[te],pred),"balanced_accuracy":balanced_accuracy_score(y[te],pred),
             "macro_f1":f1_score(y[te],pred,average="macro",zero_division=0),
             "top2_accuracy":topk(y[te],p,2),"top3_accuracy":topk(y[te],p,3)}
        row.update({f"recall_{c}":recall_score(y[te],pred,labels=[i],average="macro",zero_division=0) for i,c in enumerate(CLASS_NAMES)})
        folds.append(row)
        records.extend({"protocol":label,"fold":fold,"row":int(idx),"group_id":groups[idx],"y_true":int(y[idx]),"y_pred":int(pred[j]), **{f"p_{c}":p[j,i] for i,c in enumerate(CLASS_NAMES)}}
                       for j,idx in enumerate(te))
    return pd.DataFrame(folds),pd.DataFrame(records)

random_folds,random_oof=evaluate(StratifiedKFold(n_splits=5,shuffle=True,random_state=RANDOM_STATE),label="random_stratified_5fold")
group_folds,group_oof=evaluate(StratifiedGroupKFold(n_splits=N_SPLITS,shuffle=True,random_state=RANDOM_STATE),True,True,f"grouped_nested_{N_SPLITS}fold")
folds=pd.concat([random_folds,group_folds]); oof=pd.concat([random_oof,group_oof])
display(folds)


In [ ]:
METRICS=["accuracy","balanced_accuracy","macro_f1","top2_accuracy","top3_accuracy"]+[f"recall_{c}" for c in CLASS_NAMES]
def bootstrap_group(records, metric, n=2000):
    rng=np.random.default_rng(RANDOM_STATE); ids=records.group_id.unique(); values=[]
    for _ in range(n):
        part=pd.concat([records[records.group_id.eq(g)] for g in rng.choice(ids,len(ids),replace=True)])
        yt,yp=part.y_true.to_numpy(),part.y_pred.to_numpy()
        if metric=="accuracy": v=accuracy_score(yt,yp)
        elif metric=="balanced_accuracy": v=balanced_accuracy_score(yt,yp)
        elif metric=="macro_f1": v=f1_score(yt,yp,average="macro",zero_division=0)
        elif metric.startswith("recall_"):
            v=recall_score(yt,yp,labels=[CLASS_NAMES.index(metric[7:])],average="macro",zero_division=0)
        else: v=topk(yt,part[[f"p_{c}" for c in CLASS_NAMES]].to_numpy(),int(metric[3]))
        values.append(v)
    return np.quantile(values,[.025,.975])
summary=folds.groupby("protocol")[METRICS].agg(["mean","std"])
ci=pd.DataFrame([{"metric":m,"mean":group_folds[m].mean(),"ci_low":bootstrap_group(group_oof,m)[0],"ci_high":bootstrap_group(group_oof,m)[1]} for m in METRICS])
folds.to_csv(OUTPUT_DIR/"fold_metrics.csv",index=False,encoding="utf-8-sig"); oof.to_csv(OUTPUT_DIR/"oof_predictions.csv",index=False,encoding="utf-8-sig")
summary.to_csv(OUTPUT_DIR/"random_vs_grouped_summary.csv",encoding="utf-8-sig"); ci.to_csv(OUTPUT_DIR/"grouped_bootstrap_ci.csv",index=False,encoding="utf-8-sig")
display(summary); display(ci)
fig,axs=plt.subplots(1,2,figsize=(12,5))
for ax,title,part in zip(axs,["随机分层","分组嵌套"],[random_oof,group_oof]):
    ConfusionMatrixDisplay(confusion_matrix(part.y_true,part.y_pred,labels=range(4)),display_labels=CLASS_NAMES).plot(ax=ax,colorbar=False); ax.set_title(title)
plt.tight_layout(); plt.savefig(OUTPUT_DIR/"confusion_random_vs_grouped.png",dpi=220); plt.show()
print("完成，输出目录：",OUTPUT_DIR)
